In [17]:
import logging
import pprint
import sys
import os
import tarfile
import os.path
import boto3

import sagemaker as sage
from time import gmtime, strftime

logging.basicConfig(stream=sys.stdout, level=logging.INFO,
                        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

sess = sage.Session()
account = sess.boto_session.client("sts").get_caller_identity()["Account"]

In [18]:

def make_tarfile(output_filename, source_dir):
    with tarfile.open(output_filename, "w:gz") as tar:
        tar.add(source_dir, arcname=os.path.basename(source_dir))

        
def create_source():
    make_tarfile("/tmp/sourcedir.tar.gz", "./")
    s3 = boto3.client('s3')
    with open('/tmp/sourcedir.tar.gz', 'rb') as data:
        s3.upload_fileobj(data, f'models-nofakes', f'model_filter/sourcedir.tar.gz')
    s3_bucket = 'models-nofakes'
    model_filename = 'model_filter/sourcedir.tar.gz'
    model_data_url = f's3://{s3_bucket}/{model_filename}'
    return model_data_url

In [23]:
    name = 'initial-filters'
    environment = 'dev'
    version = 'v0'
    model_name = f"{name}-{environment}-{version}"
    model_source = create_source()
    s3_bucket = 'models-nofakes'
    model_filename = 'model_filter/sourcedir.tar.gz'
    model_data_url = f's3://{s3_bucket}/{model_filename}'
    sagemaker = boto3.client("sagemaker")
    image = "141502667606.dkr.ecr.eu-west-1.amazonaws.com/sagemaker-scikit-learn:0.20.0-cpu-py3"
    
    model_environment = {
        "SAGEMAKER_PROGRAM": "validation_filter_main.py",
        "SAGEMAKER_SUBMIT_DIRECTORY": model_source,
        "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
        "SAGEMAKER_REGION": "eu-west-1"
    }
    
    print(model_environment)

    # In case model exists, since it cannot be updated, only recreated.
    try:
        sagemaker.delete_model(ModelName=model_name)
        logger.info(f"Deleted already existing model with name: {model_name}")
    except Exception as e:
        logger.info(f"Couldn't delete existing model: {e}")

    # Creating model
    model_response = sagemaker.create_model(
        ModelName=model_name,
        ExecutionRoleArn='arn:aws:iam::719332055132:role/service-role/AmazonSageMaker-ExecutionRole-20210707T151388',
        PrimaryContainer={
            "Image": image,
            "ImageConfig": {
                "RepositoryAccessMode": "Platform"
            },
            "Mode": "SingleModel",
            "ModelDataUrl": model_data_url,
            "Environment": model_environment
        }
    )

    logger.info(model_response)

{'SAGEMAKER_PROGRAM': 'validation_filter_main.py', 'SAGEMAKER_SUBMIT_DIRECTORY': 's3://models-nofakes/model_filter/sourcedir.tar.gz', 'SAGEMAKER_CONTAINER_LOG_LEVEL': '20', 'SAGEMAKER_REGION': 'eu-west-1'}
2022-05-06 10:04:42,038 - __main__ - INFO - Couldn't delete existing model: An error occurred (ValidationException) when calling the DeleteModel operation: Could not find model "arn:aws:sagemaker:eu-west-1:719332055132:model/initial-filters-dev-v0".
2022-05-06 10:04:42,430 - __main__ - INFO - {'ModelArn': 'arn:aws:sagemaker:eu-west-1:719332055132:model/initial-filters-dev-v0', 'ResponseMetadata': {'RequestId': 'a5950141-560b-4f3f-b8eb-8d49d78c267d', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'a5950141-560b-4f3f-b8eb-8d49d78c267d', 'content-type': 'application/x-amz-json-1.1', 'content-length': '84', 'date': 'Fri, 06 May 2022 10:04:42 GMT'}, 'RetryAttempts': 0}}


In [25]:
    # create sagemaker endpoint config
    create_endpoint_config_api_response = sagemaker.create_endpoint_config(
                                                EndpointConfigName='initial-filters-endpoint-config-'+version,
                                                ProductionVariants=[
                                                    {
                                                        'VariantName': 'initial-filters-test',
                                                        'ModelName': model_name,
                                                        'InitialInstanceCount': 1,
                                                        'InstanceType': 'ml.t2.medium'
                                                    },
                                                ]
                                           )

    print ("create_endpoint_config API response", create_endpoint_config_api_response)

create_endpoint_config API response {'EndpointConfigArn': 'arn:aws:sagemaker:eu-west-1:719332055132:endpoint-config/initial-filters-endpoint-config-v0', 'ResponseMetadata': {'RequestId': '0507a929-0cf9-4c3c-acad-0d9862dfcd9f', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '0507a929-0cf9-4c3c-acad-0d9862dfcd9f', 'content-type': 'application/x-amz-json-1.1', 'content-length': '115', 'date': 'Fri, 06 May 2022 10:05:31 GMT'}, 'RetryAttempts': 0}}


In [26]:
    # create sagemaker endpoint
    create_endpoint_api_response = sagemaker.create_endpoint(
                                        EndpointName='initial-filters-endpoint-'+version,
                                        EndpointConfigName='initial-filters-endpoint-config-'+version,
                                    )

    print ("create_endpoint API response", create_endpoint_api_response)   


create_endpoint API response {'EndpointArn': 'arn:aws:sagemaker:eu-west-1:719332055132:endpoint/initial-filters-endpoint-v0', 'ResponseMetadata': {'RequestId': '97067b77-13d2-4808-a07f-1025bd960e1a', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '97067b77-13d2-4808-a07f-1025bd960e1a', 'content-type': 'application/x-amz-json-1.1', 'content-length': '95', 'date': 'Fri, 06 May 2022 10:05:48 GMT'}, 'RetryAttempts': 0}}


# INFERENCE (for try)

In [16]:
from io import BytesIO, StringIO
import json
import pandas as pd

# Try format
input_dict ={ 
    "user_id": "04", 
    "product_id": "92",
    "review": "Todo ha ido bien, le pongo 5 estrellas", 
    "score": "5"
} 
      
# Serializing json  
json_object = json.dumps(input_dict, indent = 4) 
print(json_object)

j = pd.DataFrame([json.loads(json_object)])
print(j)
 

{
    "user_id": "04",
    "product_id": "92",
    "review": "Todo ha ido bien, le pongo 5 estrellas",
    "score": "5"
}
  user_id product_id                                  review score
0      04         92  Todo ha ido bien, le pongo 5 estrellas     5


In [64]:
import json
import sagemaker
import boto3
import pandas as pd
import time
from tqdm import tqdm

endpoint_name = 'dani-test-v2-endpoint'

# Try format
input_dict ={ 
    "user_id": "04", 
    "product_id": "92",
    "review": "Todo ha ido bien, le pongo 5 estrellas", 
    "score": 5
} 
      
# Serializing json  
json_object = json.dumps(input_dict, indent = 4) 
print(json_object)

pipeline_endpoint = sagemaker.predictor.Predictor(endpoint_name)

start_time = time.time()
response = pipeline_endpoint.predict(json_object, {'ContentType': 'application/json'})

time.sleep(2)
predictions = json.loads(response)

print(predictions)

{
    "user_id": "04",
    "product_id": "92",
    "review": "Todo ha ido bien, le pongo 5 estrellas",
    "score": 5
}
{'filter_failed': '', 'motive': '', 'suspicious': False}


In [4]:
response = sagemaker.delete_endpoint(
    EndpointName='basic-filter-endpoint-v1'
)

ClientError: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Cannot update in-progress endpoint "arn:aws:sagemaker:eu-west-1:719332055132:endpoint/basic-filter-endpoint-v1".